In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

sales_bronze = spark.read.option("header", True).csv("/Volumes/retail_project/retail_db/retail_files/bronze/online_retail_sales_500k.csv")
customers_bronze = spark.read.option("header", True).csv("/Volumes/retail_project/retail_db/retail_files/bronze/customers_150k.csv")
products_bronze = spark.read.option("header", True).csv("/Volumes/retail_project/retail_db/retail_files/bronze/products_120k.csv")
inventory_bronze = spark.read.option("header", True).csv("/Volumes/retail_project/retail_db/retail_files/bronze/inventory_200k.csv")


In [0]:
sales = (
    sales_bronze
    .withColumnRenamed("InvoiceNo", "transaction_id")
    .withColumnRenamed("InvoiceDate", "invoice_date")
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("StockCode", "product_id")
    .withColumnRenamed("UnitPrice", "price")
    .withColumnRenamed("Quantity", "quantity")
    .withColumn("invoice_date", to_timestamp("invoice_date"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("quantity", col("quantity").cast("int"))
)

In [0]:
sales = (
    sales
    .filter(col("customer_id").isNotNull())
    .filter(col("price") > 0)
    .filter(col("quantity") > 0)
)


In [0]:
sales = sales.dropDuplicates(["transaction_id", "product_id"])


In [0]:
sales = sales.withColumn(
    "total_amount",
    col("quantity") * col("price")
)


In [0]:
customers = (
    customers_bronze
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)


In [0]:
w = Window.partitionBy("customer_id").orderBy(col("effective_from").desc())

customers_scd2 = (
    customers
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)


In [0]:
from pyspark.sql.functions import broadcast

sales_enriched = (
    sales
    .join(broadcast(products_bronze.withColumnRenamed("StockCode","product_id")),
          "product_id", "left")
    .join(broadcast(inventory_bronze.withColumnRenamed("StockCode","product_id")),
          "product_id", "left")
)


In [0]:
sales_enriched = sales_enriched.withColumn(
    "salt_key",
    floor(rand() * 10)
)

inventory_salted = inventory_bronze.withColumn(
    "salt_key",
    floor(rand() * 10)
)

sales_enriched = sales_enriched.join(
    broadcast(inventory_salted),
    ["product_id", "salt_key"],
    "left"
)


In [0]:
final_silver_sales = sales_enriched.select(
    "transaction_id",
    "invoice_date",
    "customer_id",
    "product_id",
    "Description",
    "quantity",
    "price",
    "total_amount",
    "stock_qty",
    "stock_status"
)


In [0]:
sales = sales.withColumnRenamed("price", "sale_price")


In [0]:
products=(
    products_bronze
    .withColumnRenamed("price", "product_price"))

In [0]:
final_silver_sales = (
    sales
    .join(customers, on="customer_id", how="left")
    .join(products, on="product_id", how="left")
)
final_silver_sales.display()

In [0]:
final_silver_sales.explain(True)
final_silver_sales.printSchema()


In [0]:
final_silver_sales = final_silver_sales.drop("Country")


In [0]:
from collections import Counter

Counter(final_silver_sales.columns)


In [0]:
silver_sales=final_silver_sales.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/retail_project/retail_db/retail_files/silver/")
